# LARRY preprocessing

Applies the QC chain `PROJECT_SUMMARY_27082026.pdf` S2.3 documents for LARRY,
on top of the raw combined matrix from `code/build_larry_matrix.py`:

0. **Cell-calling whitelist** (SPRING-style knee, tuned) -- the base
   "real cell" population everything else operates on
1. Remove cells above 20% mitochondrial content
2. Merge lineage barcodes within a Hamming distance of 3 (already applied
   upstream, in `build_larry_matrix.py`'s `extract_clones()` -- this notebook
   compares the two possible merge rules on that same output before moving on)
3. Remove multiplets with singletCode
4. Discard cells expressing fewer than 200 genes

Built independently from the PDF's own description and the public
[AllonKleinLab/LARRY](https://github.com/AllonKleinLab/LARRY) code -- not from
any other notebook in this repo or elsewhere.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.sparse as sp

import singletCode

# [2026-10-01 commented out: /scratch/gzu5140 is purged scratch; see REPOINT_LOG.tsv] PROCESSED = Path("/scratch/gzu5140/ka_twinfer/larry_dataset/processed")
from twinfer.utils.paths import get_larry_dataset_dir  # [2026-10-01 added]
PROCESSED = Path(get_larry_dataset_dir() / "processed")
QC_DIR = PROCESSED / "qc_filtered"
QC_DIR.mkdir(exist_ok=True)


## Load the raw combined matrix

In [ ]:
X = sio.mmread(PROCESSED / "larry_combined_counts.mtx").tocsr()
genes = pd.Series(Path(PROCESSED / "genes.txt").read_text().split("\n")).replace("", np.nan).dropna().reset_index(drop=True)
cells_idx = pd.Series(Path(PROCESSED / "cells.txt").read_text().split("\n")).replace("", np.nan).dropna().reset_index(drop=True)
obs = pd.read_csv(PROCESSED / "obs_metadata.csv", index_col=0)

assert X.shape == (len(cells_idx), len(genes))
assert list(obs.index) == list(cells_idx)
print(f"raw matrix: {X.shape[0]} cells x {X.shape[1]} genes, {X.nnz} nonzeros")
obs.head()


## Cell-calling whitelist (SPRING-style knee, tuned)

`clonal_annotation.ipynb` (the public LARRY repo) expects a `CELL_BCS_PATH`/
`LIB_NAMES_PATH` whitelist as input -- an ordered list of "real" cells,
corresponding to the rows of the expression matrix. That whitelist is itself
the *output* of a separate tool from the same lab,
[SPRING's inDrops data-prep pipeline](https://github.com/AllonKleinLab/SPRING_dev/blob/master/data_prep/spring_example_HPCs.ipynb):
rank barcodes by total transcript count and cut at the **knee** -- the point
where the barcode-rank curve bends from a flat, high plateau (real cells) to
a steep fall (empty-droplet/ambient noise). See the companion artifact
for a visual walkthrough of this on three of our own libraries.

A raw automatic knee (maximum distance from the chord joining the curve's two
ends, in log-log space) turns out to be too strict here -- it kept only
~48,000 of 95,584 cells, whereas `LSK_d2_d4_d6.h5ad` (the *already
scrublet-filtered* released object) alone has 72,946, and
`PROJECT_SUMMARY_27082026.pdf`'s own rebuilt input matrix has 83,735 (itself
an undercount, missing cells in `LSK_d2_1` and one other sample). Since our
whitelist sits *before* singletCode's own multiplet removal, it should
legitimately be larger than either reference, not smaller: target
**83,735 + ~2,000 = 85,735** cells.

Method: compute each library's own automatic knee threshold, then
binary-search a single shared multiplier `k` applied to all 15 thresholds at
once (`threshold_lib = knee_lib * k`) until the aggregate pass count hits the
target -- one shared "how generous" knob rather than 15 independently
re-tuned ones. (Checked separately: of `LSK_d2_d4_d6.h5ad`'s 72,946 cells,
100% exist somewhere in our raw 95,584-cell matrix, 97.8% survive this
whitelist, and the 2.2% that don't all sit at 51-100% of their library's
threshold -- a boundary effect, not something systematic.)


In [ ]:
TARGET_WHITELIST_CELLS = 83735 + 2000  # PDF's own (undercounted) rebuild + ~2k


def knee_threshold(counts):
    c = np.sort(counts)[::-1]
    c = c[c > 0]
    x = np.log10(np.arange(1, len(c) + 1))
    y = np.log10(c)
    p1, p2 = np.array([x[0], y[0]]), np.array([x[-1], y[-1]])
    line = p2 - p1
    line_norm = line / np.linalg.norm(line)
    vecs = np.stack([x - p1[0], y - p1[1]], axis=1)
    proj_len = vecs @ line_norm
    proj = np.outer(proj_len, line_norm) + p1
    dist = np.linalg.norm(vecs - (proj - p1), axis=1)
    return c[np.argmax(dist)]


total_counts = np.asarray(X.sum(axis=1)).ravel()
obs["total_counts"] = total_counts

libs = sorted(obs["library"].unique())
lib_masks = {lib: (obs["library"] == lib).to_numpy() for lib in libs}
knee_thr = {lib: float(knee_threshold(total_counts[lib_masks[lib]])) for lib in libs}


def n_pass(k):
    return sum(int((total_counts[lib_masks[lib]] >= knee_thr[lib] * k).sum()) for lib in libs)


lo, hi = 1e-4, 1.0
for _ in range(40):
    mid = (lo + hi) / 2
    if n_pass(mid) >= TARGET_WHITELIST_CELLS:
        lo = mid
    else:
        hi = mid
k = lo

whitelist_mask = np.zeros(len(obs), dtype=bool)
for lib in libs:
    m = lib_masks[lib]
    whitelist_mask |= m & (total_counts >= knee_thr[lib] * k)

print(f"shared multiplier k={k:.5f} (raw knee scaled down to {k:.1%} of its value)")
print(f"whitelist: {int(whitelist_mask.sum())} / {len(obs)} cells (target {TARGET_WHITELIST_CELLS})")


## Barcode Hamming-collapse: greedy vs transitive

`PROJECT_SUMMARY_27082026.pdf` S2.3: *"Merged within a Hamming distance of 3,
each group collapsing to its highest-UMI member: 17,076 barcodes to 11,211."*

"Group" + "highest-UMI member" describes **transitive** clustering (connected
components of the Hamming<=3 graph, canonical = most-abundant member) -- not
the public `clonal_annotation.ipynb` notebook's **greedy** rule (first
accepted representative wins, order-dependent, no highest-UMI selection at
all). `build_larry_matrix.py` computes both and uses transitive+highest-UMI as
canonical for `obs['larry_clone']`; this cell checks how much that choice
actually mattered on our own data before trusting it.


In [ ]:
reads = pd.read_csv(PROCESSED / "reads_precollapse.csv")  # library, cell, umi, gfp_bc, n_reads
greedy_map = pd.read_csv(PROCESSED / "barcode_map_greedy.csv").set_index("raw_barcode")["representative"].to_dict()
trans_map = pd.read_csv(PROCESSED / "barcode_map_transitive.csv").set_index("raw_barcode")["representative"].to_dict()

n_distinct_greedy = len(set(greedy_map.values()))
n_distinct_trans = len(set(trans_map.values()))
print(f"distinct raw barcodes: {reads['gfp_bc'].nunique()}")
print(f"distinct clones -- greedy: {n_distinct_greedy}, transitive: {n_distinct_trans}")


def clone_calls_for(bc_map, n_umis=3):
    """Same aggregation build_larry_matrix.py uses for its own (non-singletCode)
    larry_clone column, applied under a given barcode-collapse map, purely to
    compare the two collapse rules' downstream effect on a like-for-like basis."""
    r = reads.copy()
    r["clone_bc"] = r["gfp_bc"].map(bc_map)
    umi_support = (
        r.drop_duplicates(["library", "cell", "umi", "clone_bc"])
        .groupby(["library", "cell", "clone_bc"]).size().reset_index(name="n_umis")
    )
    called = umi_support[umi_support["n_umis"] >= n_umis]
    calls = called.groupby(["library", "cell"])["clone_bc"].apply(lambda s: "".join(sorted(s))).reset_index()
    calls["cell_id"] = calls["library"] + ":" + calls["cell"]
    return calls.set_index("cell_id")["clone_bc"]


calls_greedy = clone_calls_for(greedy_map)
calls_trans = clone_calls_for(trans_map)
compare = pd.DataFrame({"greedy": calls_greedy, "transitive": calls_trans}).fillna("")
n_diff = int((compare["greedy"] != compare["transitive"]).sum())
print(f"cells with a called clone under either method: {len(compare)}")
print(f"cells whose clone call differs between the two methods: {n_diff} "
      f"({n_diff / len(compare):.1%})")

# adopt transitive + highest-UMI as canonical, matching the PDF's own description
canonical_map = trans_map


## Mitochondrial filter (>20% removed)

In [ ]:
mito_mask = genes.str.lower().str.startswith("mt-")
n_mito = int(mito_mask.sum())
if n_mito == 0:
    raise RuntimeError(
        "No mitochondrial genes found via case-insensitive 'mt-' match -- refusing to "
        "silently apply a no-op filter (this is exactly the MT- vs mt- case bug "
        "PROJECT_SUMMARY_27082026.pdf documents elsewhere)."
    )
print(f"found {n_mito} mitochondrial genes: {genes[mito_mask].tolist()}")

mito_counts = np.asarray(X[:, mito_mask.values].sum(axis=1)).ravel()
pct_counts_mt = np.divide(
    mito_counts, total_counts, out=np.zeros_like(mito_counts, dtype=float), where=total_counts > 0
) * 100

pass_mito = pct_counts_mt <= 20.0
print(f"{(pass_mito & whitelist_mask).sum()} / {whitelist_mask.sum()} whitelisted cells "
      f"pass the <=20% mitochondrial filter")


## Multiplet removal via singletCode

`singletCode.get_singlets()` expects a `cellID, barcode, sample` sample sheet
with UMI support encoded as repeated rows (it derives `nUMI` itself via
`groupby(['cellID','barcode','sample']).size()`) -- exactly the shape
`umi_table.csv` was built in. `cellID` here is the **bare** real barcode (no
library prefix) and `sample` is the library, so singletCode's own
`get_multilane_barcodes` cross-sample check can see genuine barcode reuse
across libraries (confirmed real, not a hashing artifact, at ~37% of distinct
barcodes) -- library-prefixing the cellID would have silently disabled that
check entirely.

Restricted to the whitelist above (same reasoning as the mito filter: this is
the "real cell" population everything else acts on). Cells with no row at all
in `umi_table.csv` have no barcode record and are not kept either, per the PDF
("cells with no barcode record cannot be judged and are not kept").


In [ ]:
umi_table = pd.read_csv(PROCESSED / "umi_table.csv")  # cellID, barcode, sample (exact column order singletCode requires)

whitelist_keys = set(zip(obs.loc[whitelist_mask, "library"], cells_idx[whitelist_mask].str.split(":", n=1).str[1]))
umi_table = umi_table[[(s, c) in whitelist_keys for s, c in zip(umi_table["sample"], umi_table["cellID"])]].reset_index(drop=True)
print(f"umi_table restricted to the whitelist: {len(umi_table)} rows")

singletCode.check_sample_sheet(umi_table)
good_data, singlet_stats = singletCode.get_singlets(umi_table, dataset_name="LARRY")
display(singlet_stats)


In [ ]:
singlets = good_data[good_data["label"] == "Singlet"]
singlet_keys = set(zip(singlets["sample"], singlets["cellID"]))

obs_library = obs["library"].to_numpy()
obs_barcode = np.array([c.split(":", 1)[1] for c in cells_idx])
is_singlet = np.array(
    [(lib, bc) in singlet_keys for lib, bc in zip(obs_library, obs_barcode)]
)
print(f"{is_singlet.sum()} / {len(is_singlet)} cells labeled Singlet "
      f"(cells with no barcode record at all, or outside the whitelist, are excluded here too)")

# the clone barcode(s) singletCode actually kept per singlet cell, for the final obs table
singlet_clone = (
    singlets.groupby(["sample", "cellID"])["barcode"]
    .apply(lambda s: "".join(sorted(set(s))))
)
singlet_clone.index = [f"{lib}:{bc}" for lib, bc in singlet_clone.index]


## Gene-count floor (<200 genes discarded)

In [ ]:
n_genes_per_cell = np.asarray((X > 0).sum(axis=1)).ravel()
pass_genes = n_genes_per_cell >= 200
print(f"{(pass_genes & whitelist_mask).sum()} / {whitelist_mask.sum()} whitelisted cells "
      f"pass the >=200-gene floor")


## Combine and save

In [ ]:
n0 = int(whitelist_mask.sum())
after_mito = whitelist_mask & pass_mito
n1 = int(after_mito.sum())
after_singlet = after_mito & is_singlet
n2 = int(after_singlet.sum())
after_genes = after_singlet & pass_genes
n3 = int(after_genes.sum())

print("Per-step cell counts (order matches PROJECT_SUMMARY_27082026.pdf S2.3, "
      "starting from the whitelist rather than the full raw matrix):")
print(f"  input (raw)              {len(obs)}")
print(f"  after whitelist          {n0}")
print(f"  after mito filter        {n1}")
print(f"  after singletCode        {n2}")
print(f"  after gene floor (final) {n3}")

keep = after_genes
keep_idx = np.flatnonzero(keep)
X_qc = X[keep_idx]
kept_cells = cells_idx[keep].reset_index(drop=True)

obs_qc = obs[keep].copy()
obs_qc["pct_counts_mt"] = pct_counts_mt[keep]
obs_qc["n_genes"] = n_genes_per_cell[keep]
obs_qc["singlet_label"] = np.where(is_singlet[keep], "Singlet", "Multiplet")
obs_qc["larry_clone_singletcode"] = obs_qc.index.map(singlet_clone).fillna("")

assert X_qc.shape[0] == len(obs_qc) == len(kept_cells)
assert obs_qc.index.is_unique

sio.mmwrite(QC_DIR / "larry_qc_counts.mtx", X_qc)
(QC_DIR / "genes.txt").write_text("\n".join(genes) + "\n")
(QC_DIR / "cells.txt").write_text("\n".join(kept_cells) + "\n")
obs_qc.to_csv(QC_DIR / "obs_metadata.csv")

summary = {
    "n_raw_cells": len(obs),
    "n_after_whitelist": n0,
    "n_after_mito": n1,
    "n_after_singletcode": n2,
    "n_after_gene_floor": n3,
    "whitelist_target": TARGET_WHITELIST_CELLS,
    "whitelist_shared_multiplier_k": k,
    "n_distinct_clones_greedy": n_distinct_greedy,
    "n_distinct_clones_transitive": n_distinct_trans,
    "n_cells_clone_call_differs_between_methods": n_diff,
}
(QC_DIR / "preprocessing_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))
